# Notebook 00b — Clean Extraction

**Purpose:** Extract one furnace's data, apply all cleaning cuts, save to Databricks Delta table.

```
Raw Databricks data
  → detect cracking runs (via HC feed drop)
  → keep only COMPLETE cycles (decoke confirmed on both sides)
  → remove: decoking periods (feed ≈ 0)
  → remove: warm-up at run start (feed stepping up, COT unstable)
  → remove: pre-decoke tail at run end (feed winding down)
  → save to Delta table in Databricks (never to local disk)
```


## 1. Parameters


In [0]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 10,
})

# ── Repo root: 2 levels up from notebooks/00b_clean/ ─────────────────────
REPO_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('.')))
sys.path.insert(0, REPO_ROOT)

# ── Load .env for local VSCode runs (no-op inside Databricks) ────────────
_env = os.path.join(REPO_ROOT, '.env')
if os.path.exists(_env):
    with open(_env) as _f:
        for _line in _f:
            _line = _line.strip()
            if _line and not _line.startswith('#') and '=' in _line:
                _k, _, _v = _line.partition('=')
                os.environ.setdefault(_k.strip(), _v.strip())

from olefins_ddf.io_events import get_spark
from olefins_ddf import catalog as cat, features as feat_mod
from olefins_ddf.runs import segment_runs, cracking_mask

# ── Parameters (injected by papermill for batch runs) ─────────────────────
FURNACE    = '1HA'
START      = '2025-02-05'
END        = '2026-03-20'
BUCKET_MIN = 30

# ── Delta catalog: <unity_catalog>.<schema> ───────────────────────────────
DELTA_CATALOG = 'indorama_corporate_olefins_paas_azure_weu_dev_research.workspaces'

# ── Output dir for PNG + CSV only (no time-series data written locally) ───
OUTPUT_DIR = os.path.join(REPO_ROOT, 'output', FURNACE, '00b_clean')

# ── Analysis window thresholds ────────────────────────────────────────────
FEED_SETTLED_FRAC = 0.95   # feed > 95% run median → feed is up
COT_STD_THRESH_C  = 2.0    # rolling std(COT) < 2°C over 4h → COT settled
COT_ROLL_STEPS    = 8      # 8 × 30 min = 4 h
MIN_WARMUP_H      = 12.0   # always skip first 12h (ΔΔ baseline window)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Furnace    : {FURNACE}')
print(f'Window     : {START} → {END}')
print(f'Bucket     : {BUCKET_MIN} min')
print(f'REPO_ROOT  : {REPO_ROOT}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')


## 2. Connect to Databricks & load tag catalog


In [0]:
spark   = get_spark()                     # uses DDF_CLUSTER_ID env var
catalog = cat.build_catalog(spark)        # from cache unless refresh=True

# Ensure research schema exists (no-op if already exists)
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {DELTA_CATALOG}')
print(f'Schema ready   : {DELTA_CATALOG}')

tube_tags = catalog[catalog['role'] == 'tube_COT']
if 'furnace' in tube_tags.columns:
    tube_tags = tube_tags[tube_tags['furnace'] == FURNACE]
print(f'Catalog loaded : {len(catalog)} total tags')
print(f'tube_COT       : {len(tube_tags)} tags for {FURNACE} (expect 192)')


## 3. Load feature matrix

Load from Delta cache if available, otherwise build from raw historian (~5–10 min).
**No time-series data is ever written to local disk.**


In [0]:
DELTA_TABLE_FEAT = f'{DELTA_CATALOG}.{FURNACE.lower()}_features'

try:
    print(f'Loading from Delta cache: {DELTA_TABLE_FEAT}')
    feat = spark.table(DELTA_TABLE_FEAT).toPandas()
    # Index column may be named 'timestamp' or 'ts' depending on how it was saved
    ts_col = next((c for c in ['timestamp', 'ts'] if c in feat.columns), None)
    if ts_col is None:
        raise ValueError(f'No timestamp column in Delta table. Got: {list(feat.columns)}')
    feat = feat.set_index(ts_col)
    feat.index = pd.to_datetime(feat.index)
    feat.index.name = 'timestamp'
    print(f'Loaded {len(feat):,} rows from Delta cache.')
except Exception as e:
    print(f'Cache miss ({e}). Building from historian (~5–10 min)...')
    feat, _ = feat_mod.build_feature_matrix(
        spark, catalog, FURNACE, start=START, end=END, bucket=BUCKET_MIN
    )
    # Normalise index name before saving — load_wide returns 'ts', we store as 'timestamp'
    feat.index.name = 'timestamp'
    (spark.createDataFrame(feat.reset_index())
       .write.format('delta')
       .mode('overwrite')
       .option('overwriteSchema', 'true')
       .saveAsTable(DELTA_TABLE_FEAT))
    print(f'Saved to Delta: {DELTA_TABLE_FEAT}')

print(f'Feature matrix : {feat.shape[0]:,} rows × {feat.shape[1]} cols')
print(f'Time range     : {feat.index.min()} → {feat.index.max()}')
print(f'Index name     : {feat.index.name}  (must be timestamp, not ts or integer)')


## 4. Detect cracking runs — keep complete cycles only

**Complete cycle** = a cracking run where a decoke was observed **before** it started
AND **after** it ended (both within our data window).

First and last runs are almost always incomplete — the data window cuts them.


In [0]:
feed_col = 'feed_total' if 'feed_total' in feat.columns else None
if feed_col is None:
    raise ValueError(f'feed_total not found. Available: {list(feat.columns)}')

mask     = cracking_mask(feat[feed_col])
runs_all = segment_runs(mask)

# Keep only COMPLETE cycles:
# - there must be at least one non-cracking timestep BEFORE this run (= prior decoke seen)
# - there must be at least one non-cracking timestep AFTER this run (= next decoke seen)
runs = []
for r in runs_all:
    pre  = mask[mask.index < r.start]
    post = mask[mask.index > r.end]
    prior_decoke = len(pre) > 0 and (~pre).any()
    post_decoke  = len(post) > 0 and (~post).any()
    if prior_decoke and post_decoke:
        runs.append(r)

print(f'All runs detected       : {len(runs_all)}')
print(f'Complete cycles (kept)  : {len(runs)}  — decoke confirmed on both sides')
print(f'Incomplete (cut)        : {len(runs_all) - len(runs)}  — boundary runs excluded')
print()
for r in runs:
    print(f'  Run {r.index:2d}: {r.start.date()} → {r.end.date()}  ({r.length_days:.1f} d)')


## 5. Find analysis window for each run

Inside each run, find the stable cracking window:
- **Start:** feed > 95% median AND rolling std(COT) < 2°C/4h AND t ≥ run.start + 12h
- **End:** last timestamp where feed > 95% median (cuts pre-decoke tail)


In [0]:
def find_analysis_window(run, feat_df):
    seg = feat_df.loc[run.start:run.end].copy()
    null = dict(run=run.index, run_start=run.start, run_end=run.end,
                analysis_start=pd.NaT, analysis_end=pd.NaT,
                warmup_hours=np.nan, tail_hours=np.nan,
                length_days=round(run.length_days, 2),
                clean_days=np.nan, settled=False)
    if seg.empty:
        return null

    earliest_start = run.start + pd.Timedelta(hours=MIN_WARMUP_H)

    # Criterion 1: feed at operating level
    run_med = seg['feed_total'][seg['feed_total'] > 0].median()
    feed_ok = seg['feed_total'] > FEED_SETTLED_FRAC * run_med

    # Criterion 2: COT settled
    cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in seg.columns), None)
    if cot_col:
        cot_ok = seg[cot_col].rolling(COT_ROLL_STEPS, min_periods=2).std() < COT_STD_THRESH_C
    else:
        cot_ok = pd.Series(True, index=seg.index)

    # Criterion 3: minimum time past run start
    time_ok = seg.index >= earliest_start

    settled_idx = (feed_ok & cot_ok & time_ok)
    settled_idx = settled_idx[settled_idx].index

    if settled_idx.empty:
        analysis_start = earliest_start
        settled = False
    else:
        analysis_start = settled_idx[0]
        settled = True

    # End = last point where feed is still at operating level (cuts pre-decoke tail)
    end_idx = feed_ok[feed_ok].index
    analysis_end = end_idx[-1] if not end_idx.empty else run.end
    if analysis_end <= analysis_start:
        analysis_end = run.end

    warmup_hours = (analysis_start - run.start).total_seconds() / 3600
    tail_hours   = (run.end - analysis_end).total_seconds() / 3600
    clean_days   = (analysis_end - analysis_start).total_seconds() / 86400

    return dict(
        run=run.index, run_start=run.start, run_end=run.end,
        analysis_start=analysis_start, analysis_end=analysis_end,
        warmup_hours=round(warmup_hours, 1), tail_hours=round(tail_hours, 1),
        length_days=round(run.length_days, 2), clean_days=round(clean_days, 2),
        settled=settled,
    )


win_df = pd.DataFrame([find_analysis_window(r, feat) for r in runs])

print(f'Windows computed : {len(win_df)} runs')
print(f'  Settled        : {win_df["settled"].sum()} / {len(win_df)}')
print(f'  Warmup avg     : {win_df["warmup_hours"].mean():.1f} h  '
      f'(range {win_df["warmup_hours"].min():.0f}–{win_df["warmup_hours"].max():.0f} h)')
print(f'  Tail avg       : {win_df["tail_hours"].mean():.1f} h')
print(f'  Clean avg      : {win_df["clean_days"].mean():.1f} d')
display(win_df[['run','run_start','run_end','analysis_start','analysis_end',
                'warmup_hours','tail_hours','clean_days','settled']])


## 6. Build clean time series

Keep only rows inside valid analysis windows. Everything else is removed:
decoking, warm-up, pre-decoke tail, and incomplete cycles.


In [0]:
clean_mask  = pd.Series(False, index=feat.index)
run_id_col  = pd.Series(np.nan,  index=feat.index)
valid_runs  = win_df[win_df['analysis_start'].notna() & win_df['analysis_end'].notna()].copy()

for _, w in valid_runs.iterrows():
    a_start, a_end = w['analysis_start'], w['analysis_end']
    if pd.isna(a_start) or pd.isna(a_end) or a_end <= a_start:
        continue
    in_win = (feat.index >= a_start) & (feat.index <= a_end)
    clean_mask[in_win] = True
    run_id_col[in_win] = int(w['run'])

feat_clean = feat[clean_mask].copy()
feat_clean['run_id'] = run_id_col[clean_mask].astype(int)
feat_clean['phase']  = 'clean'

total_rows = len(feat)
clean_rows = len(feat_clean)
pct_kept   = 100 * clean_rows / total_rows

print(f'Total rows           : {total_rows:,}')
print(f'Clean rows kept      : {clean_rows:,}  ({pct_kept:.1f}%)')
print(f'Rows removed         : {total_rows - clean_rows:,}  ({100-pct_kept:.1f}%)')
print(f'Complete runs        : {feat_clean["run_id"].nunique()}')
print(f'Clean span           : {feat_clean.index.min()} → {feat_clean.index.max()}')
print(f'Total clean time     : {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')


## 7. Manual exclusions

Harry's principle: only remove physically impossible sensor events, not real high-ΔΔ periods.
Add entries after DCS verification.


In [0]:
# Format: (reason, start_timestamp, end_timestamp)
MANUAL_EXCLUDE = [
    # ('Run9_sensor_fault', '2025-10-28', '2025-11-02'),  # ~120°C spike — verify with DCS first
]

excluded_count = 0
for reason, t0, t1 in MANUAL_EXCLUDE:
    exc = (feat_clean.index >= t0) & (feat_clean.index <= t1)
    if 'dd_abs_max' in feat_clean.columns:
        feat_clean.loc[exc, 'dd_abs_max'] = np.nan
    excluded_count += exc.sum()
    print(f'  Excluded {exc.sum()} rows: {reason}')

if not MANUAL_EXCLUDE:
    print('No manual exclusions applied.')
else:
    print(f'Total manually excluded: {excluded_count} rows')


## 8. Save clean data to Delta table

All data stays in Databricks. Only PNG plot and run-windows CSV (~15 rows) saved locally.


In [0]:
DELTA_TABLE_CLEAN = f'{DELTA_CATALOG}.{FURNACE.lower()}_clean'

feat_clean.index.name = 'timestamp'   # ensure consistent name
(spark.createDataFrame(feat_clean.reset_index())
    .write.format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(DELTA_TABLE_CLEAN))

print(f'Clean data saved : {DELTA_TABLE_CLEAN}')
print(f'Shape            : {feat_clean.shape}')

# Run windows metadata — small CSV, OK locally
win_path = os.path.join(OUTPUT_DIR, 'run_analysis_windows.csv')
win_df.to_csv(win_path, index=False)
print(f'Run windows CSV  : {win_path}  (metadata only, ~15 rows)')


## 9. Verification plot


In [0]:
fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
fig.suptitle(f'{FURNACE} — Clean extraction (green=kept · red=removed)',
             fontsize=12, fontweight='bold')

ax1, ax2, ax3 = axes

# Feed
if 'feed_total' in feat.columns:
    ax1.plot(feat.index, feat['feed_total'], color='#378add', lw=0.6, alpha=0.7, label='raw')
    ax1.plot(feat_clean.index, feat_clean['feed_total'], color='#1d9e75', lw=0.8, label='clean')
ax1.set_ylabel('Feed total (NM³/H)')
ax1.legend(fontsize=8, loc='upper right')

# COT
cot_col = next((c for c in ['cot', 'cot_ctrl'] if c in feat.columns), None)
if cot_col:
    ax2.plot(feat.index, feat[cot_col], color='#7f77dd', lw=0.6, alpha=0.7, label='raw')
    ax2.plot(feat_clean.index, feat_clean[cot_col], color='#085041', lw=0.8, label='clean')
ax2.set_ylabel('COT (°C)')
ax2.legend(fontsize=8, loc='upper right')

# ΔΔ
if 'dd_abs_max' in feat.columns:
    ax3.plot(feat.index, feat['dd_abs_max'], color='#ef9f27', lw=0.6, alpha=0.5, label='raw')
    ax3.plot(feat_clean.index, feat_clean['dd_abs_max'], color='#d85a30', lw=0.9, label='clean')
    ax3.axhline(45, color='#e24b4a', ls='--', lw=1.0, label='Alarm 45°C')
    ax3.axhline(30, color='#ef9f27', ls='--', lw=1.0, label='Pre-alarm 30°C')
ax3.set_ylabel('dd_abs_max (°C)')
ax3.legend(fontsize=8, loc='upper right')

# Shade windows
for _, w in valid_runs.iterrows():
    for ax in axes:
        ax.axvspan(w['run_start'],      w['analysis_start'], alpha=0.12, color='#e24b4a', zorder=0)
        ax.axvspan(w['analysis_start'], w['analysis_end'],   alpha=0.10, color='#1d9e75', zorder=0)
        ax.axvspan(w['analysis_end'],   w['run_end'],        alpha=0.12, color='#e24b4a', zorder=0)

plt.tight_layout()
plot_path = os.path.join(OUTPUT_DIR, f'{FURNACE}_00b_clean_extraction.png')
plt.savefig(plot_path, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')


## 10. Summary


In [0]:
print('=' * 60)
print(f'CLEAN EXTRACTION SUMMARY — {FURNACE}')
print('=' * 60)
print(f'  Raw window       : {START} → {END}')
print(f'  Total runs found : {len(runs_all)}')
print(f'  Complete cycles  : {len(runs)}  (decoke on both sides)')
print(f'  Clean rows kept  : {clean_rows:,}  ({pct_kept:.1f}%)')
print(f'  Total clean time : {clean_rows * BUCKET_MIN / 60 / 24:.1f} days')
print()
print('Storage (all in Databricks — no time-series data locally):')
print(f'  Features   : {DELTA_TABLE_FEAT}')
print(f'  Clean data : {DELTA_TABLE_CLEAN}')
print(f'  Run CSV    : {win_path}  (metadata only)')
print(f'  PNG plot   : {plot_path}')
print()
print('How to use downstream:')
print(f'  df = spark.table("{DELTA_TABLE_CLEAN}").toPandas()')
print( '  df = df.set_index("timestamp")')
print( '  # All rows are clean — use run_id to iterate runs')
print()

suspicious = win_df[(win_df['warmup_hours'] > 48) | (win_df['tail_hours'] > 48)]
if len(suspicious):
    print('⚠  Verify with DCS (warmup or tail > 48h):')
    for _, s in suspicious.iterrows():
        print(f'   Run {int(s["run"]):2d}: warmup={s["warmup_hours"]:.0f}h  '
              f'tail={s["tail_hours"]:.0f}h  ({s["run_start"].date()} → {s["run_end"].date()})')
